# Train the `pick_egg` ACT policy on Google Colab

Trains the pick policy for the Dino egg catch challenge from the `CreatorKanata/dino_pick_egg` dataset and pushes it to
`CreatorKanata/act_dino_pick_egg`. The Mac application downloads that checkpoint later and runs it from the catch pose.

**Prerequisites**

- Google Colab **Pro** or **Pro+** with a GPU runtime (Runtime > Change runtime type > T4, L4 or A100). On Pro+, background execution keeps training running after the browser tab is closed; the Slack notifications below report start and end. The fork needs Python 3.12 or newer; the setup cell checks this.
- A Hugging Face token with **write** access stored as a Colab secret named `HF_TOKEN` (key icon in the left sidebar, notebook access enabled).
- The dataset on the Hub: `CreatorKanata/dino_pick_egg`, LeRobot v3 format at 30 fps, cameras `front` and `wrist` at 640x480, arm and base state and action keys as recorded by `lekiwi_client`. It is recorded by the application's recorder (the application docs also describe this fork's `lerobot-record`), each episode running from the catch pose until the egg is held; see `docs/lekiwi-app-development.md` section 6 in the application repository.
- Google Drive, used as the checkpoint store that survives a Colab disconnect.
- Optional: a Slack Incoming Webhook stored as the Colab secret `SLACK_WEBHOOK_URL`, for messages when training starts, finishes, or fails, and when the export cell pushes the model. To create it: at https://api.slack.com/apps choose Create New App > From scratch, pick the workspace, open **Incoming Webhooks**, switch it on, click **Add New Webhook to Workspace**, choose the channel, and copy the `https://hooks.slack.com/services/...` URL into a Colab secret named `SLACK_WEBHOOK_URL` with notebook access enabled. Without the secret the notebook prints a note and skips the messages.

**What it produces**

- An ACT policy trained on the **wrist camera only** plus joint state (set `CAMERAS = ["front", "wrist"]` for the two-camera baseline), pushed to the Hub at the end of training.
- Checkpoints mirrored to `/content/drive/MyDrive/dino_pick_egg/<run name>/checkpoints/`, the training log at `.../train.log`, and a marker file `DONE-<UTC stamp>.txt` or `FAILED-<UTC stamp>.txt` with the run summary when a run ends.
- The held-out loss on 10 percent of the episodes per task, a loss curve, and the input feature names the application's policy runner must provide.

**How camera selection works (verified in this fork, 2026-09-25).** `DatasetConfig` has no per-camera filter, but `lerobot-train` accepts `--policy.input_features` from the command line. `make_policy` in `src/lerobot/policies/factory.py` only derives the input features from the dataset when they are empty, and `validate_visual_features_consistency` accepts a policy that uses a subset of the dataset's cameras. The training cell therefore passes the dataset's features filtered to `CAMERAS` (option (a) of the plan, a CLI override). The dataloader still decodes the unused camera; the model ignores it.

**Why training writes to local disk first.** `lerobot-train` marks the newest checkpoint with a `checkpoints/last` symlink, and the Google Drive mount in Colab does not support symlinks. Training therefore writes to `/content/outputs/<run name>` and the notebook copies each completed checkpoint to Drive while it runs.

**Run order.** Setup > Helpers > Configuration > Dataset inspection > then **either** Train (fresh run) **or** Resume (after a disconnect: re-run Setup, Helpers, Configuration first) > Offline evaluation > Export. Edit the Configuration cell before running.

**Expected duration.** Setup takes about 5 minutes. Training time depends on the GPU and on video decoding; for about 60 episodes of 20 s and the default 12 epochs, expect a few hours (roughly 2 h on an A100 and longer on a T4). This is an estimate, not a measurement: read the steps per second from the log after the first minutes and adjust `EPOCHS` if needed.

---
## 1. Setup

GPU check, install of this fork, Hugging Face login, and Google Drive. The fork repository is public (checked 2026-09-25), so no GitHub token is needed. If pip replaces Colab's preinstalled `torch`, Colab may ask to restart the runtime: restart, then continue with the next cell.

In [ ]:
import subprocess
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"The fork requires Python 3.12+ (pyproject.toml requires-python); this runtime has {sys.version}"
    )

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU found. Use Runtime > Change runtime type and pick a GPU (T4, L4 or A100).")
print(result.stdout)

In [ ]:
# `training` = `dataset` (datasets, torchcodec, av) + `accelerate` (required by lerobot-train) + wandb.
%pip install -q "lerobot[training] @ git+https://github.com/CreatorKanata/lerobot-dino-egg-catch-challenge@main"

# If the fork is ever made private, store a GitHub token (read access) as the Colab secret GH_TOKEN, then:
# from google.colab import userdata
# GH_TOKEN = userdata.get("GH_TOKEN")
# REPO = "github.com/CreatorKanata/lerobot-dino-egg-catch-challenge"
# %pip install -q "lerobot[training] @ git+https://{GH_TOKEN}@{REPO}@main"

In [ ]:
import os

from google.colab import drive, userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Colab secret HF_TOKEN is missing or empty; add a token with write access.")

# The environment variable is inherited by the lerobot-train subprocess (Hub download and push).
os.environ["HF_TOKEN"] = HF_TOKEN
login = subprocess.run(["hf", "auth", "login", "--token", HF_TOKEN], capture_output=True, text=True)
if login.returncode != 0:
    raise RuntimeError(f"hf auth login failed: {login.stderr.strip()}")
print(subprocess.run(["hf", "auth", "whoami"], capture_output=True, text=True).stdout)

drive.mount("/content/drive")

---
## 2. Helpers

Functions used by the cells below. The first cell covers feature selection, the held-out split and checkpoint mirroring to Drive. The second runs `lerobot-train` with streamed output and sends the Slack notifications (`notify` posts `{"text": ...}` to the webhook with a 10 s timeout and never raises).

In [ ]:
import json
import math
import os
import re
import shutil
import subprocess
import time
from pathlib import Path

from lerobot.configs import FeatureType
from lerobot.utils.feature_utils import dataset_to_policy_features

CAMERA_PREFIX = "observation.images."
PROGRESS_PRINT_INTERVAL_S = 60
MIRROR_CHECK_INTERVAL_S = 30


def policy_input_features(meta, cameras: list[str]) -> dict:
    # Dataset features in policy form (images as C,H,W), minus actions and the unused cameras.
    features = dataset_to_policy_features(meta.features)
    wanted = {CAMERA_PREFIX + name for name in cameras}
    missing = wanted - set(meta.camera_keys)
    if missing:
        raise ValueError(f"Cameras {sorted(missing)} are not in the dataset; it has {meta.camera_keys}")
    return {
        key: {"type": ft.type.value, "shape": list(ft.shape)}
        for key, ft in features.items()
        if ft.type is not FeatureType.ACTION and (ft.type is not FeatureType.VISUAL or key in wanted)
    }


def held_out_episodes(meta, eval_split: float) -> list[int]:
    # Same rule as make_train_eval_datasets in src/lerobot/datasets/factory.py:
    # the last ceil(n * eval_split) episodes of each task are held out.
    by_task: dict[str, list[int]] = {}
    for ep in range(meta.total_episodes):
        tasks = meta.episodes[ep]["tasks"]
        key = tasks[0] if tasks else ""
        by_task = {**by_task, key: [*by_task.get(key, []), ep]}
    held = []
    for eps in by_task.values():
        n_eval = math.ceil(len(eps) * eval_split)
        held = [*held, *eps[len(eps) - n_eval :]]
    return sorted(held)


def step_dirs(checkpoints_dir: Path) -> list[Path]:
    # Completed step directories (numeric names), oldest first.
    if not checkpoints_dir.is_dir():
        return []
    dirs = [p for p in checkpoints_dir.iterdir() if p.is_dir() and p.name.isdigit()]
    return sorted(dirs, key=lambda p: int(p.name))


def mirror_latest_checkpoint(local_dir: Path, drive_dir: Path, keep: int) -> Path | None:
    # Copy the checkpoint that `checkpoints/last` points at to Drive. lerobot-train moves the link only after
    # the step directory is fully written, so the copy never sees a half-written checkpoint.
    last = local_dir / "checkpoints" / "last"
    if not last.is_symlink():
        return None
    source = last.resolve()
    target = drive_dir / "checkpoints" / source.name
    if target.exists():
        return target
    partial = target.with_name(target.name + ".partial")
    shutil.rmtree(partial, ignore_errors=True)
    shutil.copytree(source, partial)
    partial.rename(target)
    for old in step_dirs(drive_dir / "checkpoints")[:-keep]:
        shutil.rmtree(old, ignore_errors=True)
    print(f"[mirror] checkpoint {source.name} copied to {target}", flush=True)
    return target

In [ ]:
import collections
import dataclasses
import datetime as dt

import requests

TRAIN_LINE = re.compile(r"step:\S+ smpl:\S+ ep:\S+ epch:([\d.]+) loss:([\d.]+)")
EVAL_LINE = re.compile(r"step (\d+): eval_loss=([\d.]+)")
CHECKPOINT_LINE = re.compile(r"Checkpoint policy after step (\d+)")
PROGRESS_LINE = re.compile(r"\|\s*(\d+)/\d+")
SLACK_TIMEOUT_S = 10
TAIL_LINES = 8
SLACK_TAIL_CHARS = 1500


@dataclasses.dataclass(frozen=True)
class TrainingOutcome:
    returncode: int | None  # None when the runner itself failed
    last_step: int
    train_loss: float | None
    eval_loss: float | None
    tail: tuple[str, ...]
    error: str | None = None


def notify(text: str) -> None:
    # Post to the Slack webhook from the Configuration cell; never raises.
    url = globals().get("SLACK_WEBHOOK_URL")
    if not url:
        print(f"[slack skipped] {text}", flush=True)
        return
    try:
        response = requests.post(url, json={"text": text}, timeout=SLACK_TIMEOUT_S)
        if response.status_code != 200:
            print(f"[slack] HTTP {response.status_code}: {response.text[:200]}", flush=True)
    except Exception as exc:  # noqa: BLE001 - a notification must never stop training
        print(f"[slack] post failed: {exc!r}", flush=True)


def gpu_name() -> str:
    try:
        query = ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
        out = subprocess.run(query, capture_output=True, text=True, timeout=10)
        return out.stdout.strip().splitlines()[0] if out.returncode == 0 and out.stdout.strip() else "unknown"
    except Exception:  # noqa: BLE001
        return "unknown"


def hours_minutes(seconds: float) -> str:
    return f"{int(seconds // 3600)}:{int(seconds % 3600 // 60):02d}"


def utc_stamp() -> str:
    return dt.datetime.now(dt.UTC).strftime("%Y%m%dT%H%M%SZ")


def write_marker(drive_dir: Path, kind: str, text: str) -> None:
    # DONE-<stamp>.txt or FAILED-<stamp>.txt on Drive; never raises.
    try:
        drive_dir.mkdir(parents=True, exist_ok=True)
        marker = drive_dir / f"{kind}-{utc_stamp()}.txt"
        marker.write_text(text + "\n")
        print(f"Marker written: {marker}", flush=True)
    except Exception as exc:  # noqa: BLE001
        print(f"Could not write the {kind} marker: {exc!r}", flush=True)


def run_training(
    cmd: list[str], local_dir: Path, drive_dir: Path, keep: int, start_step: int
) -> TrainingOutcome:
    # Run lerobot-train, stream its output, append it to train.log on Drive, mirror new checkpoints.
    # Returns what happened instead of raising, so the caller can always send a notification.
    drive_dir.mkdir(parents=True, exist_ok=True)
    print(" \\\n    ".join(cmd), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    last_progress = last_mirror = 0.0
    last_step, train_loss, eval_loss = start_step, None, None
    tail = collections.deque(maxlen=TAIL_LINES)
    proc = None
    try:
        with open(drive_dir / "train.log", "a") as log:
            proc = subprocess.Popen(
                cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env
            )
            for line in proc.stdout:
                now = time.monotonic()
                if line.startswith("Training:"):  # tqdm progress bar: print once a minute, do not log
                    if match := PROGRESS_LINE.search(line):
                        last_step = max(last_step, start_step + int(match.group(1)))
                    if now - last_progress > PROGRESS_PRINT_INTERVAL_S:
                        print(line.rstrip(), flush=True)
                        last_progress = now
                else:
                    print(line, end="", flush=True)
                    log.write(line)
                    tail.append(line.rstrip())
                    if match := TRAIN_LINE.search(line):
                        train_loss = float(match.group(2))
                    if match := EVAL_LINE.search(line):
                        eval_loss = float(match.group(2))
                        last_step = max(last_step, int(match.group(1)))
                    if match := CHECKPOINT_LINE.search(line):
                        last_step = max(last_step, int(match.group(1)))
                if now - last_mirror > MIRROR_CHECK_INTERVAL_S:
                    log.flush()
                    mirror_latest_checkpoint(local_dir, drive_dir, keep)
                    last_mirror = now
            returncode = proc.wait()
        mirror_latest_checkpoint(local_dir, drive_dir, keep)
        return TrainingOutcome(returncode, last_step, train_loss, eval_loss, tuple(tail))
    except BaseException as exc:  # includes KeyboardInterrupt (Runtime > Interrupt)
        if proc is not None and proc.poll() is None:
            proc.terminate()
            proc.wait(timeout=60)
        return TrainingOutcome(None, last_step, train_loss, eval_loss, tuple(tail), error=repr(exc))


def train_with_notifications(
    cmd: list[str], *, steps: int, batch_size: int, repo_id: str, cameras: list[str], start_step: int = 0
) -> TrainingOutcome:
    # Slack message and Drive marker at start, success, and failure of one lerobot-train run.
    label = f" (resumed from step {start_step})" if start_step else ""
    notify(
        f"dino_pick_egg training started{label}: {DATASET_REPO_ID} ({meta.total_episodes} ep, "
        f"{meta.total_frames} frames), policy {repo_id}, cameras {cameras}, {steps} steps, "
        f"batch {batch_size}, GPU {gpu_name()}"
    )
    started = time.monotonic()
    try:
        outcome = run_training(cmd, LOCAL_OUTPUT_DIR, OUTPUT_DIR, KEEP_CHECKPOINTS_ON_DRIVE, start_step)
    except Exception as exc:  # noqa: BLE001 - run_training returns errors, but never lose the message
        outcome = TrainingOutcome(None, start_step, None, None, (), error=repr(exc))
    elapsed = hours_minutes(time.monotonic() - started)

    if outcome.returncode == 0 and outcome.error is None:
        loss = f"{outcome.train_loss:.4f}" if outcome.train_loss is not None else "n/a"
        held_out = f", held-out eval loss {outcome.eval_loss:.4f}" if outcome.eval_loss is not None else ""
        summary = (
            f"dino_pick_egg training finished{label} in {elapsed}: {steps} steps, "
            f"final train loss {loss}{held_out}, pushed to {repo_id}"
        )
        notify(summary)
        write_marker(OUTPUT_DIR, "DONE", summary)
        return outcome

    reason = outcome.error or f"lerobot-train exited with code {outcome.returncode}"
    tail = "\n".join(outcome.tail)[-SLACK_TAIL_CHARS:]
    summary = f"dino_pick_egg training FAILED{label} after {elapsed} at step {outcome.last_step}: {reason}"
    if tail:
        summary = f"{summary}\n```{tail}```"
    notify(summary)
    write_marker(OUTPUT_DIR, "FAILED", summary)
    raise RuntimeError(f"{reason}; see {OUTPUT_DIR / 'train.log'}")

---
## 3. Configuration

Edit this cell, then run it. `STEPS` follows the steps and epochs rule in `AGENT_GUIDE.md` section 7.2:
`steps_per_epoch = ceil(total_frames / batch_size)`, `steps = epochs x steps_per_epoch`. The guide's typical range is 5 to 10 epochs; this run uses about 12 (owner choice). ACT has no LR scheduler preset, so no scheduler scaling is needed.

In [ ]:
from google.colab import userdata

from lerobot.datasets import LeRobotDatasetMetadata

DATASET_REPO_ID = "CreatorKanata/dino_pick_egg"
POLICY_TYPE = "act"
CAMERAS = ["wrist"]  # plan: wrist only; baseline comparison: ["front", "wrist"]

# The baseline gets its own repo so it never overwrites the wrist-only policy the application uses.
WRIST_ONLY_REPO_ID = "CreatorKanata/act_dino_pick_egg"
POLICY_REPO_ID = WRIST_ONLY_REPO_ID if CAMERAS == ["wrist"] else f"{WRIST_ONLY_REPO_ID}_{'_'.join(CAMERAS)}"
RUN_NAME = f"{POLICY_TYPE}_{'_'.join(CAMERAS)}"

BATCH_SIZE = 8  # AGENT_GUIDE.md 7.3: 8-16 for ACT
EPOCHS = 12
SAVE_FREQ = 2_000  # checkpoint every N steps: a disconnect loses at most this many steps
EVAL_STEPS = SAVE_FREQ  # held-out loss every N steps (needs EVAL_SPLIT > 0)
LOG_FREQ = 100
EVAL_SPLIT = 0.1  # last 10 percent of the episodes of each task are held out
WANDB = False
KEEP_CHECKPOINTS_ON_DRIVE = 2  # each ACT checkpoint (weights + optimizer state) is several hundred MB

# Optional Slack Incoming Webhook (Colab secret SLACK_WEBHOOK_URL); see the header for how to create it.
try:
    SLACK_WEBHOOK_URL = userdata.get("SLACK_WEBHOOK_URL") or None
except Exception:  # secret missing or notebook access not granted
    SLACK_WEBHOOK_URL = None

DRIVE_ROOT = Path("/content/drive/MyDrive/dino_pick_egg")
OUTPUT_DIR = DRIVE_ROOT / RUN_NAME  # Drive: mirrored checkpoints and train.log
LOCAL_OUTPUT_DIR = Path("/content/outputs") / RUN_NAME  # lerobot-train --output_dir (local disk)

meta = LeRobotDatasetMetadata(DATASET_REPO_ID)
steps_per_epoch = math.ceil(meta.total_frames / BATCH_SIZE)
STEPS = EPOCHS * steps_per_epoch

print(f"Dataset        : {DATASET_REPO_ID} ({meta.total_episodes} episodes, {meta.total_frames} frames)")
print(f"Steps per epoch: {steps_per_epoch} at batch size {BATCH_SIZE}")
print(f"Steps          : {STEPS} (~{EPOCHS} epochs), checkpoint every {SAVE_FREQ}")
print(f"Cameras        : {CAMERAS}")
print(f"Policy         : {POLICY_TYPE} -> {POLICY_REPO_ID}")
print(f"Local output   : {LOCAL_OUTPUT_DIR}")
print(f"Drive output   : {OUTPUT_DIR}")
slack_state = "on" if SLACK_WEBHOOK_URL else "off (no SLACK_WEBHOOK_URL secret; messages are skipped)"
print(f"Slack          : {slack_state}")

---
## 4. Dataset inspection

Check the episode count, the features, and the tasks, and look at one `wrist` and one `front` frame from episode 0: the egg should be visible in the wrist view and the colors should look natural (not swapped channels).

In [ ]:
import matplotlib.pyplot as plt

from lerobot.datasets import LeRobotDataset

print(f"Episodes: {meta.total_episodes}  Frames: {meta.total_frames}  FPS: {meta.fps}")
print(f"Robot type: {meta.robot_type}")
print("\nFeatures:")
for name, ft in meta.features.items():
    print(f"  {name:32s} {ft['dtype']:8s} {tuple(ft['shape'])}")
print("\nTasks:")
for task in meta.tasks.index:
    n_eps = sum(1 for ep in range(meta.total_episodes) if task in meta.episodes[ep]["tasks"])
    print(f"  {task}  ({n_eps} episodes)")

episode0 = LeRobotDataset(DATASET_REPO_ID, episodes=[0])  # downloads only the files that hold episode 0
frame = episode0[0]
shown = [key for key in (CAMERA_PREFIX + "wrist", CAMERA_PREFIX + "front") if key in frame]
fig, axes = plt.subplots(1, len(shown), figsize=(6 * len(shown), 4.5), squeeze=False)
for ax, key in zip(axes[0], shown, strict=True):
    ax.imshow(frame[key].permute(1, 2, 0).numpy())  # C,H,W float in [0, 1] -> H,W,C
    ax.set_title(f"{key} (episode 0, frame 0)")
    ax.axis("off")
plt.show()
print(f"Task of episode 0: {frame['task']}")

---
## 5. Train (fresh run)

Runs `lerobot-train` with the flags below; every flag was checked against this fork's source:

| Flag | Defined in |
| --- | --- |
| `--dataset.repo_id`, `--dataset.eval_split` | `DatasetConfig` in `src/lerobot/configs/default.py` |
| `--policy.type=act` | `ACTConfig` registered as `act` in `src/lerobot/policies/act/configuration_act.py` |
| `--policy.device`, `--policy.repo_id`, `--policy.push_to_hub`, `--policy.input_features` | `PreTrainedConfig` in `src/lerobot/configs/policies.py` |
| `--output_dir`, `--job_name`, `--batch_size`, `--steps`, `--save_freq`, `--log_freq`, `--eval_steps` | `TrainPipelineConfig` in `src/lerobot/configs/train.py` |
| `--wandb.enable` | `WandBConfig` in `src/lerobot/configs/default.py` |

**Camera selection: CLI override of the policy input features** (option (a)). `--policy.input_features` gets the dataset's state feature and only the cameras in `CAMERAS`, with shapes read from the dataset metadata. The saved `config.json` keeps exactly these features, so the application sees `observation.images.wrist` and `observation.state` as the model inputs.

`--policy.push_to_hub=true` pushes the final model, pre- and post-processors, and `train_config.json` to `POLICY_REPO_ID` when training ends. Slack gets a message when the run starts, when it finishes (duration, final train loss, held-out loss, repo), and when it fails or the cell is interrupted (step and the last log lines); a `DONE-` or `FAILED-<UTC stamp>.txt` marker with the same summary is written to `OUTPUT_DIR`. If the runtime disconnects, use section 6 instead of re-running this cell.

In [ ]:
if LOCAL_OUTPUT_DIR.exists():
    raise FileExistsError(f"{LOCAL_OUTPUT_DIR} already exists: use the Resume cell, or change RUN_NAME.")
if step_dirs(OUTPUT_DIR / "checkpoints"):
    raise FileExistsError(f"{OUTPUT_DIR} already holds checkpoints: use the Resume cell, or change RUN_NAME.")

INPUT_FEATURES = policy_input_features(meta, CAMERAS)
print("Policy input features:", json.dumps(INPUT_FEATURES, indent=2))

train_cmd = [
    "lerobot-train",
    f"--dataset.repo_id={DATASET_REPO_ID}",
    f"--dataset.eval_split={EVAL_SPLIT}",
    f"--policy.type={POLICY_TYPE}",
    "--policy.device=cuda",
    f"--policy.repo_id={POLICY_REPO_ID}",
    "--policy.push_to_hub=true",
    f"--policy.input_features={json.dumps(INPUT_FEATURES)}",
    f"--output_dir={LOCAL_OUTPUT_DIR}",
    f"--job_name={RUN_NAME}",
    f"--batch_size={BATCH_SIZE}",
    f"--steps={STEPS}",
    f"--save_freq={SAVE_FREQ}",
    f"--log_freq={LOG_FREQ}",
    f"--eval_steps={EVAL_STEPS}",
    f"--wandb.enable={str(WANDB).lower()}",
]
train_with_notifications(
    train_cmd, steps=STEPS, batch_size=BATCH_SIZE, repo_id=POLICY_REPO_ID, cameras=CAMERAS
)

---
## 6. Resume after a disconnect

After a disconnect, run Setup, Helpers and Configuration again (same `CAMERAS`), then this cell. It takes the newest checkpoint from local disk or Drive, restores it to the original local run directory with its `last` link, and runs
`lerobot-train --config_path=<checkpoint>/pretrained_model/train_config.json --resume=true` (`resume` in `TrainPipelineConfig`; `config_path` is read by `src/lerobot/configs/parser.py`). The resumed run takes all settings from the checkpoint's `train_config.json`, including the camera selection, the step count and the Hub push. The Slack messages and markers work as in section 5 and are marked "resumed from step N"; a runtime that disconnects cannot send the failure message, so a missing end message means: check Colab and resume.

In [ ]:
candidates = step_dirs(LOCAL_OUTPUT_DIR / "checkpoints") + step_dirs(OUTPUT_DIR / "checkpoints")
if not candidates:
    raise FileNotFoundError(
        f"No checkpoint in {LOCAL_OUTPUT_DIR} or {OUTPUT_DIR}; start with the Train cell."
    )
latest = max(candidates, key=lambda p: int(p.name))
local_step_dir = LOCAL_OUTPUT_DIR / "checkpoints" / latest.name
if not local_step_dir.exists():
    shutil.copytree(latest, local_step_dir)
last_link = LOCAL_OUTPUT_DIR / "checkpoints" / "last"
if last_link.is_symlink() or last_link.exists():
    last_link.unlink()
last_link.symlink_to(latest.name)

train_config = local_step_dir / "pretrained_model" / "train_config.json"
saved = json.loads(train_config.read_text())
saved_output_dir = Path(saved["output_dir"])
if saved_output_dir != LOCAL_OUTPUT_DIR:
    raise ValueError(
        f"Checkpoint was trained into {saved_output_dir}, not {LOCAL_OUTPUT_DIR}: check CAMERAS and RUN_NAME."
    )
print(f"Resuming from step {int(latest.name)} ({latest})")
resume_cmd = ["lerobot-train", f"--config_path={train_config}", "--resume=true"]
saved_features = saved["policy"]["input_features"]
saved_cameras = [key.removeprefix(CAMERA_PREFIX) for key in saved_features if key.startswith(CAMERA_PREFIX)]
train_with_notifications(
    resume_cmd,
    steps=saved["steps"],
    batch_size=saved["batch_size"],
    repo_id=saved["policy"]["repo_id"],
    cameras=saved_cameras,
    start_step=int(latest.name),
)

---
## 7. Offline evaluation

The fork's offline evaluation is the held-out loss that `lerobot-train` computes every `EVAL_STEPS` steps on the `EVAL_SPLIT` episodes (`eval_steps` and `dataset.eval_split`, logged as `eval_loss`). There is no separate offline evaluation command for a real-robot dataset (`lerobot-eval` runs simulation environments). This cell plots the training and held-out loss from `train.log` and lists the held-out episodes. Real success, the egg held at the end, is measured on the robot.

In [ ]:
import matplotlib.pyplot as plt

held = held_out_episodes(meta, EVAL_SPLIT)
held_frames = sum(meta.episodes[ep]["length"] for ep in held)
train_frames = meta.total_frames - held_frames
print(f"Held-out episodes ({len(held)}): {held}")
print(f"Training frames: {train_frames}, held-out frames: {held_frames}")

log_text = (OUTPUT_DIR / "train.log").read_text()
train_points = [(float(e), float(loss)) for e, loss in TRAIN_LINE.findall(log_text)]
eval_points = [(int(s) * BATCH_SIZE / train_frames, float(loss)) for s, loss in EVAL_LINE.findall(log_text)]
if not train_points:
    raise ValueError(f"No training loss lines in {OUTPUT_DIR / 'train.log'} yet.")

plt.figure(figsize=(8, 4.5))
plt.plot(*zip(*train_points), label="train loss")
if eval_points:
    plt.plot(*zip(*eval_points), "o-", label="held-out loss")
plt.xlabel("epoch")
plt.ylabel("loss (L1 + KL weight x KLD)")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print(f"Last train loss: {train_points[-1][1]:.4f}")
if eval_points:
    print(f"Last held-out loss: {eval_points[-1][1]:.4f}")

---
## 8. Export

Training already pushes the final model when it completes. This cell pushes the newest checkpoint's `pretrained_model/` only if the Hub repo has no model yet (for example, when the run was stopped early), then prints what the Mac application needs.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
repo_files = api.list_repo_files(POLICY_REPO_ID) if api.repo_exists(POLICY_REPO_ID) else []
if "model.safetensors" in repo_files:
    print(f"{POLICY_REPO_ID} already holds a model (pushed by lerobot-train); nothing to upload.")
else:
    checkpoints = step_dirs(LOCAL_OUTPUT_DIR / "checkpoints") + step_dirs(OUTPUT_DIR / "checkpoints")
    if not checkpoints:
        raise FileNotFoundError("No checkpoint to export.")
    newest = max(checkpoints, key=lambda p: int(p.name))
    if int(newest.name) < STEPS:
        print(f"Warning: exporting step {int(newest.name)} of {STEPS}; training did not finish.")
    api.create_repo(POLICY_REPO_ID, exist_ok=True)
    api.upload_folder(
        folder_path=str(newest / "pretrained_model"),
        repo_id=POLICY_REPO_ID,
        commit_message=f"Upload {RUN_NAME} checkpoint {newest.name}",
    )
    print(f"Uploaded {newest / 'pretrained_model'} to {POLICY_REPO_ID}")
    notify(f"dino_pick_egg model pushed: https://huggingface.co/{POLICY_REPO_ID}")

policy_config = json.loads(Path(hf_hub_download(POLICY_REPO_ID, "config.json")).read_text())
print(f"\nPolicy for the Mac application: --policy.path={POLICY_REPO_ID}")
print("(the saved device is cuda; on the Mac pass --policy.device=mps)")
print("\nModel inputs (the policy runner must provide these keys and shapes):")
for key, ft in policy_config["input_features"].items():
    print(f"  {key:32s} {ft['type']:7s} {tuple(ft['shape'])}")
print("Model outputs:")
for key, ft in policy_config["output_features"].items():
    print(f"  {key:32s} {ft['type']:7s} {tuple(ft['shape'])}")
print(f"\nchunk_size={policy_config['chunk_size']}  n_action_steps={policy_config['n_action_steps']}")
print(f"Dataset fps (the runner's control rate): {meta.fps}")

---
## 9. Next steps

- **Application side:** build the policy runner in the application against the inputs printed above (`observation.images.wrist` as 3x480x640 plus `observation.state`), loading `--policy.path=CreatorKanata/act_dino_pick_egg`. It runs from the catch pose until the egg is held and must stop when `Stop` is pressed. Until then Auto Catch keeps its stub.
- **Robot evaluation:** success is the egg held at the end of the episode; compare against the teleoperated demonstrations and between checkpoints.
- **Baseline:** set `CAMERAS = ["front", "wrist"]` and run again; it pushes to `CreatorKanata/act_dino_pick_egg_front_wrist` and keeps its own Drive folder.
- **SmolVLA comparison (later, out of scope here):** same dataset and camera selection with `POLICY_TYPE = "smolvla"`, which needs the `smolvla` extra, fine-tuning from `--policy.path=lerobot/smolvla_base`, a smaller batch, and the unfrozen vision encoder (`AGENT_GUIDE.md` sections 6 and 7.6). This notebook does not cover it yet.